# 📘 IITM Java Course — Week 1, Lecture 3: Memory Management (Stack, Heap & GC)

**Instructor**: Prof. Madhavan Mukund (Chennai Mathematical Institute / IIT Madras)  
**Course**: Programming Concepts Using Java (B.Sc in Programming and Data Science)  

--- 

## 📖 Key Technical Terms Dictionary (Defined Upon First Appearance)

* **Stack Memory**: A contiguous, LIFO (Last-In, First-Out) memory region dedicated to tracking method execution frames and primitive local variables.
* **Heap Memory**: A dynamic RAM pool used for allocating objects and arrays that outlive method execution frames.
* **Activation Record (Stack Frame)**: A structured memory block pushed onto the call stack for every method invocation, containing parameters, local variables, a Control Link, and a Return Value Link.
* **Control Link (Dynamic Link)**: A memory pointer inside an activation record pointing to the caller's previous stack frame.
* **Return Value Link**: A memory location slot in the caller's stack frame where a returning function stores its result.
* **`malloc()`**: A manual C library function (`memory allocation`) that requests dynamic memory blocks from the Heap.
* **`free()`**: A manual C library function that deallocates heap memory and returns it to the free memory pool.
* **Memory Leak**: A critical defect where allocated heap memory is abandoned without being deallocated, draining available RAM over time.
* **Garbage Collection (GC)**: An automatic runtime process in Java/Python that scans the Heap and reclaims memory occupied by unreachable objects.
* **Mark-and-Sweep**: A 2-phase GC algorithm that marks all reachable objects starting from root references, then sweeps (deallocates) unmarked objects.

## 1. Defining Stack Memory vs. Heap Memory from First Principles

During program execution, memory is divided into two distinct regions to handle different variable lifecycles:

### A. What is Stack Memory?
* **Definition**: A continuous block of memory managed in a **Last-In, First-Out (LIFO)** manner, specifically dedicated to tracking function calls and local variables.
* **What it Stores**: Local primitive variables (`int`, `double`), method parameters, and object reference pointers (`myList`).
* **Allocation & Deallocation**: Automatically managed by the execution call stack. When a method is called, its frame is pushed onto the stack; when the method returns, its frame is popped and memory is immediately reclaimed.

### B. What is Heap Memory?
* **Definition**: A large pool of dynamic memory used for objects and arrays whose size or lifecycle cannot be predicted at compile time.
* **What it Stores**: Instantiated objects (`new Developer()`) and arrays (`new int[100]`).
* **Allocation & Deallocation**: Dynamically allocated using `new`. Heap memory persists **beyond** method execution exits. Deallocated via manual `free` (in C) or automatic Garbage Collection (in Java/Python).

```
+-------------------------------------------------------+
| STACK MEMORY (Grows Downward, LIFO)                   |
| - Activation Records / Stack Frames                   |
| - Primitive Local Variables (int x = 10)              |
| - Object Reference Pointers (List ref ------------------+
+-------------------------------------------------------+|  |
|                       ...                             ||  |
|                  (Free Space)                         ||  |
|                       ...                             ||  |
+-------------------------------------------------------+|  |
| HEAP MEMORY (Grows Upward, Dynamic Objects)           ||  |
| - Instantiated Objects (new Account()) <--------------+  |
| - Dynamic Lists & Arrays                              |  |
| - Persists until Garbage Collection reclaims it       |  |
+-------------------------------------------------------+  |
```

> ⚠️ **Important Distinction**: Operating system **Heap Memory** is a region of dynamic RAM. It has **no relation** to the Heap Data Structure (Binary Heap) used for Priority Queues!

## 💡 Prof. Mukund's Words of Wisdom & Best Practices (Lecture 3)

> 🌟 **1. The Water Tank Analogy for Memory Leaks**:
> * *"Think of your free memory space as a water tank. Every time you ask for dynamic storage, you open a tap and draw water. A memory leak means water is emptying out without being replenished back to the tank. Over time, large systems like web browsers or operating systems slow down or crash because they leak memory resources."*
>
> 🌟 **2. Reference Passing Warning: In-Place Mutation vs Reassignment**:
> * *"When passing objects or arrays to functions by reference, remember: you can mutate the object in-place (`arr[0] = x`) to produce intentional side-effects outside. However, if you reassign the parameter inside (`arr = new_arr`), you only change what the local parameter variable points to—it will NOT update the caller's reference variable outside!"*
>
> 🌟 **3. Garbage Collection Pause Trade-off**:
> * *"Automatic Garbage Collection removes the headache of manual memory management. But remember the tradeoff: GC runs in the background and may cause unexpected slowdowns outside your control because execution must pause during sweep cycles."*

## 2. Scope vs. Lifetime of Variables

* **Scope**: The static region of program text where a variable name can be legally accessed.
* **Lifetime**: The dynamic duration during program execution that storage remains allocated for a variable in memory.

> 💡 **Hole in Scope**: A variable's lifetime can exceed its scope. When function $f$ calls function $g$, local variables of $f$ remain allocated on $f$'s stack frame, but enter a temporary hole in scope until $g$ finishes and pops.

## 3. Activation Records (Stack Frames)

Each function invocation pushes an **Activation Record** containing:
1. **Local Variables & Parameters**: Memory for parameter values and local block variables.
2. **Control Link (Dynamic Link)**: Pointer to the caller's activation record at the bottom of the stack.
3. **Return Value Link**: Pointer indicating where to store the returned result in the calling function's frame.

In [ ]:
public class CallStackDemo {
    public static int factorial(int n) {
        if (n <= 0) {
            return 1; // Base case: pops frame from stack and returns result
        }
        return n * factorial(n - 1); // Pushes new frame on stack
    }

    public static void main(String[] args) {
        int result = factorial(4);
        System.out.println("Factorial(4) = " + result);
    }
}

CallStackDemo.main(new String[]{});

## 4. Parameter Passing: Call-by-Value vs. Call-by-Reference

### Call-by-Value (Primitives)
* A standalone copy of the primitive value is passed to the parameter slot. Changes inside the function do not affect the caller.

### Call-by-Reference / Reference-Passing (Objects & Arrays)
* The reference (pointer address) is passed.
* **In-Place Mutation**: Mutating internal properties (`arr[0] = 777`) alters the shared heap object (side-effect).
* **Reference Reassignment**: Reassigning `arr = new int[]` changes the local parameter pointer only, leaving the caller's reference unaffected.

In [ ]:
import java.util.Arrays;

public class ParameterPassingDemo {
    public static void modifyPrimitive(int x) {
        x = 999; // Modifies local copy only
    }

    public static void mutateArray(int[] arr) {
        arr[0] = 777; // Mutates shared heap memory
    }

    public static void reassignArray(int[] arr) {
        arr = new int[]{100, 200, 300}; // Points to fresh heap allocation
    }

    public static void main(String[] args) {
        int num = 10;
        modifyPrimitive(num);
        System.out.println("Primitive after modify: " + num); // 10

        int[] myArr = {1, 2, 3};
        mutateArray(myArr);
        System.out.println("Array after mutate: " + Arrays.toString(myArr)); // [777, 2, 3]

        reassignArray(myArr);
        System.out.println("Array after reassign: " + Arrays.toString(myArr)); // [777, 2, 3]
    }
}

ParameterPassingDemo.main(new String[]{});

## 5. Memory Reclamation: Manual vs. Automatic Garbage Collection

### Manual Memory Management (C/C++)
* Programmer calls `malloc()` / `free()`. Failure to free causes **Memory Leaks** (water tank running empty).

### Automatic Garbage Collection (Java & Python)
* **Mark-and-Sweep Algorithm**:
  1. **Mark**: Traverses references starting from active Stack frames (Root Set). Marks every reachable object on the Heap.
  2. **Sweep**: Scans the Heap and reclaims memory for all **unmarked** (unreachable) objects back to the free memory pool.

## 6. Comprehensive Summary & Key Takeaways
1. **Stack Memory**: Stores activation records, local primitives, and reference pointers in a LIFO manner.
2. **Heap Memory**: Stores dynamic objects and arrays; persists beyond function execution.
3. **Garbage Collection**: Mark-and-Sweep reclaims unreachable heap objects automatically.